In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append(".")
print("Ready")

Ready


In [2]:
!pip install -q langgraph-checkpoint-sqlite
print("✅ langgraph-checkpoint-sqlite installed")

✅ langgraph-checkpoint-sqlite installed



[notice] A new release of pip is available: 25.1.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
from graph.build_graph import build_graph

os.makedirs("data", exist_ok=True)
db_path = "data/checkpoints.db"

connection = sqlite3.connect(db_path, check_same_thread=False)
checkpointer = SqliteSaver(connection)
checkpointer.setup()  # creates tables if they don't exist yet; safe to call repeatedly

app = build_graph(checkpointer=checkpointer, interrupt_before=["report_generator"])
config = {"configurable": {"thread_id": "persistent-session-1"}}

initial_state = {
    "query": "What are the main safety risks of solid-state EV batteries?",
    "plan": [], "next_agent": "", "iteration_count": 0, "needs_more_research": False,
    "search_results": [], "extracted_facts": [], "fact_check_results": [], "summaries": [],
    "errors": [], "final_report": None, "model_usage_log": [],
}

result = app.invoke(initial_state, config=config)
state_now = app.get_state(config)
print("Paused before:", state_now.next)
print(f"Summaries ready: {len(state_now.values['summaries'])}")
print(f"\nDatabase file created: {os.path.exists(db_path)}, size: {os.path.getsize(db_path)} bytes")

2026-10-28 18:50:02,469 | INFO | research_system | [supervisor_agent] plan_generated sub_questions=4
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


⚠️ Gemini unavailable (ClientError), falling back to Tavily...
⚠️ Gemini unavailable (ClientError), falling back to Tavily...


2026-10-28 18:50:06,955 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=3 retries=0
2026-10-28 18:50:07,350 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=3 retries=0


⚠️ Gemini unavailable (ClientError), falling back to Tavily...
⚠️ Gemini unavailable (ClientError), falling back to Tavily...


2026-10-28 18:50:09,701 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=3 retries=0
2026-10-28 18:50:11,032 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=3 retries=0
2026-10-28 18:50:11,036 | INFO | research_system | [supervisor_agent] routing decision=data_extraction iteration=2
2026-10-28 18:50:12,026 | INFO | research_system | [data_extraction_agent] extraction_done sub_question=How do solid electrolyte interfaces degrade under thermal stress and what are the resulting failure modes? facts_found=12
2026-10-28 18:50:12,467 | INFO | research_system | [data_extraction_agent] extraction_done sub_question=What are the specific mechanical risks associated with dendrite formation in solid-state batteries during high-rate charging? facts_found=10
2026-10-28 18:50:12,670 | INFO | research_system | [data_extraction_agent] extraction_done sub_question=How does the thermal runaway behavior of solid-state b

Paused before: ('report_generator',)
Summaries ready: 3

Database file created: True, size: 4096 bytes


In [4]:

connection2 = sqlite3.connect(db_path, check_same_thread=False)
checkpointer2 = SqliteSaver(connection2)

app2 = build_graph(checkpointer=checkpointer2, interrupt_before=["report_generator"])
same_config = {"configurable": {"thread_id": "persistent-session-1"}}  # same thread_id as before

recovered_state = app2.get_state(same_config)
print("Recovered state exists:", recovered_state.values is not None)
print("Recovered 'next':", recovered_state.next)
print(f"Recovered {len(recovered_state.values['summaries'])} summaries from a completely new connection")

assert recovered_state.next == ("report_generator",)
assert len(recovered_state.values["summaries"]) > 0
print("\n State genuinely persisted to disk — survived a full 'process restart' simulation")

Recovered state exists: True
Recovered 'next': ('report_generator',)
Recovered 3 summaries from a completely new connection

 State genuinely persisted to disk — survived a full 'process restart' simulation


In [5]:
final_result = app2.invoke(None, config=same_config)
print("Report generated after simulated restart:", len(final_result["final_report"]), "chars")
print("\n" + final_result["final_report"][:300] + "...")

2026-10-28 18:54:34,922 | INFO | research_system | [report_generator_agent] report_done method=llm sub_topics=3 total_claims=37
2026-10-28 18:54:34,930 | INFO | research_system | [supervisor_agent] pipeline_complete iteration=6


Report generated after simulated restart: 3372 chars

# Safety Risks of Solid-State EV Batteries

Solid-state batteries are often touted as a safer alternative to traditional lithium-ion cells, yet they present distinct safety challenges arising from their unique material properties and manufacturing requirements. This report outlines the primary safet...


In [6]:
config_b = {"configurable": {"thread_id": "persistent-session-2"}}

state_b = app2.invoke({
    "query": "What is CATL's sodium-ion battery strategy?",
    "plan": [], "next_agent": "", "iteration_count": 0, "needs_more_research": False,
    "search_results": [], "extracted_facts": [], "fact_check_results": [], "summaries": [],
    "errors": [], "final_report": None, "model_usage_log": [],
}, config=config_b)

# confirm thread 1's state is untouched
state_a_check = app2.get_state(same_config)
print("Thread 1 still complete:", state_a_check.values["final_report"] is not None)
print("Thread 2 paused independently:", app2.get_state(config_b).next)

2026-10-28 18:55:43,121 | INFO | research_system | [supervisor_agent] plan_generated sub_questions=5


⚠️ Gemini unavailable (ClientError), falling back to Tavily...
⚠️ Gemini unavailable (ClientError), falling back to Tavily...
 Tavily unavailable (ConnectionError), falling back to DuckDuckGo...


2026-10-28 18:55:45,883 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=3 retries=0
2026-10-28 18:55:46,312 | INFO | research_system | [web_search_agent] search_done status=ok provider=duckduckgo sources=3 retries=0


⚠️ Gemini unavailable (ClientError), falling back to Tavily...
⚠️ Gemini unavailable (ClientError), falling back to Tavily...


2026-10-28 18:55:49,262 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=3 retries=0


⚠️ Gemini unavailable (ClientError), falling back to Tavily...


2026-10-28 18:55:50,592 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=3 retries=0
2026-10-28 18:55:52,487 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=3 retries=0
2026-10-28 18:55:52,492 | INFO | research_system | [supervisor_agent] routing decision=data_extraction iteration=2
2026-10-28 18:55:52,952 | INFO | research_system | [data_extraction_agent] extraction_done sub_question=What are the specific technical specifications and performance metrics of CATL's current sodium-ion battery products? facts_found=6
2026-10-28 18:55:53,567 | INFO | research_system | [data_extraction_agent] extraction_done sub_question=How does CATL's sodium-ion battery technology compare to its lithium-ion offerings in terms of cost, energy density, and cycle life? facts_found=11
2026-10-28 18:55:54,077 | INFO | research_system | [data_extraction_agent] extraction_done sub_question=Which specific market segments and ap

Thread 1 still complete: True
Thread 2 paused independently: ('report_generator',)


In [7]:
with open(".gitignore", "a") as f:
    f.write("data/checkpoints.db\n")
print("data/checkpoints.db added to .gitignore")

data/checkpoints.db added to .gitignore


In [8]:
with open("requirements.txt", "a") as f:
    f.write("langgraph-checkpoint-sqlite\n")
print(" requirements.txt updated")

 requirements.txt updated


In [ ]:
%%writefile tests/test_checkpointing.py
import os
import sqlite3
import agents.supervisor_agent as sup


def test_state_persists_across_separate_connections(tmp_path, monkeypatch):
    import graph.build_graph as gb
    from langgraph.checkpoint.sqlite import SqliteSaver

    monkeypatch.setattr(sup, "generate_plan", lambda q: ["sub-q 1"])
    monkeypatch.setattr(gb, "search_single_node", lambda state: {
        "search_results": [{"sub_question": state["query"], "status": "ok"}],
        "model_usage_log": [{"agent": "web_search_agent"}]})
    monkeypatch.setattr(gb, "data_extraction_node", lambda state: {
        "extracted_facts": [{"claim": "x"}], "model_usage_log": [{"agent": "data_extraction_agent"}]})
    monkeypatch.setattr(gb, "fact_checking_node", lambda state: {
        "fact_check_results": [{"claim": "x", "status": "CONFIRMED"}],
        "model_usage_log": [{"agent": "fact_checking_agent"}]})
    monkeypatch.setattr(gb, "summarization_node", lambda state: {
        "summaries": [{"sub_question": "q1", "summary": "a summary"}],
        "model_usage_log": [{"agent": "summarization_agent"}]})
    monkeypatch.setattr(gb, "report_generator_node", lambda state: {
        "final_report": "the report", "model_usage_log": [{"agent": "report_generator_agent"}]})

    db_path = str(tmp_path / "test_checkpoints.db")
    config = {"configurable": {"thread_id": "test-thread"}}
    initial_state = {
        "query": "test", "plan": [], "next_agent": "", "iteration_count": 0,
        "needs_more_research": False, "search_results": [], "extracted_facts": [],
        "fact_check_results": [], "summaries": [], "errors": [], "final_report": None,
        "model_usage_log": [],
    }

    conn1 = sqlite3.connect(db_path, check_same_thread=False)
    cp1 = SqliteSaver(conn1)
    cp1.setup()
    app1 = gb.build_graph(checkpointer=cp1, interrupt_before=["report_generator"])
    app1.invoke(initial_state, config=config)
    conn1.close()

    conn2 = sqlite3.connect(db_path, check_same_thread=False)
    cp2 = SqliteSaver(conn2)
    app2 = gb.build_graph(checkpointer=cp2, interrupt_before=["report_generator"])

    recovered = app2.get_state(config)
    assert recovered.next == ("report_generator",)
    assert len(recovered.values["summaries"]) == 1

    final = app2.invoke(None, config=config)
    assert final["final_report"] == "the report"